# Ayudantía 02: Programación Orientada a Objetos Avanzado 🔮
Por sus queridos ayudantes de catedra:

* Diego Toledo
* Francisca Cares
* Carlos Martel
* Agustín Becker
* Julio Huerta

## Antes que todo ...
Como saben, este curso utiliza la metodología **Flipped Classroom**. Esto significa que:

1. Antes de venir a la ayudantía o a clases, deben haber leido los contenidos
2. La ayudantía **No es una clase teórica**
3. **Aplicaremos lo que ya estudiaron** mediante resolución de ejercicios


### Hoy veremos
1. Herencia 
2. Polimorfismo
3. Clases Abstractas
4. Decoradores

## Intento de Influencer 📷

Las redes sociales son hoy en día una de las formas más populares de compartir contenido, expresarse y alcanzar visibilidad pública. Después de ver a varios amigos hacerse virales, decides que tú también quieres ser **influencer**, pero no estás seguro si abrir tu cuenta en **Instagram**, **YouTube** o **TikTok**.

A continuación, se te entrega la clase base `Usuario`, que representa a una persona en una red social. Esta clase ya está implementada y tiene los siguientes atributos:

- `user`: nombre del usuario.
- `comentarios`: cantidad de comentarios diarios que realiza.
- `likes`: cantidad de likes diarios que entrega.

Tu tarea es modelar tres tipos de influencers:

- `Instagramer`: además tiene un atributo `shares`, que representa cuántas veces sus publicaciones fueron compartidas.
- `Youtuber`: tiene un atributo `suscriptores`, que indica cuántos seguidores tiene su canal.
- `TikToker`: tiene un atributo `views`, que indica cuántas reproducciones han tenido sus videos.

Cada influencer debe tener un método `alcance()`, que calcula cuán visible es su contenido en la red. Este método se implementa de forma distinta en cada red social:

###  Cálculo del alcance basal:
- **Instagramer**: `comentarios * 2 + likes / 3`
- **Youtuber**: `likes + comentarios`
- **TikToker**: `likes * 4 - comentarios`

###  Modificaciones por baja popularidad:
- Si un **Instagramer** tiene menos de 150 `shares`, su alcance será el valor **negativo** del cálculo basal.
- Si un **Youtuber** tiene menos de 500 `suscriptores`, su alcance será la **mitad** del cálculo basal.
- Si un **TikToker** tiene menos de 300 `views`, su alcance será el cálculo basal **triplicado y negativo**.


---

Tu objetivo es implementar estas clases correctamente, aplicando:

- ✅ **Herencia** desde la clase `Usuario`.
- ✅ Uso de `super()` en los constructores.
- ✅ **Polimorfismo** sobrescribiendo el método `alcance()` de forma distinta en cada red social.

In [1]:
class Usuario:
    def __init__(self, user, comentarios, likes):
        self.user = user
        self.comentarios = comentarios
        self.likes = likes

    def __str__(self):
        return f"Usuario: {self.user}"

In [2]:
class Instagramer(Usuario):
    def __init__(self, user, comentarios, likes, shares):
        super().__init__(user, comentarios, likes)
        self.shares = shares

    def alcance(self):
        # Cálculo basal
        alcance = self.comentarios * 2 + self.likes / 3
        # Penalización por baja popularidad
        if self.shares < 150:
            return -alcance
        return alcance

In [3]:
class Youtuber(Usuario):
    def __init__(self, user, comentarios, likes, suscriptores):
        super().__init__(user, comentarios, likes)
        self.suscriptores = suscriptores

    def alcance(self):
        # Cálculo basal
        alcance = self.likes + self.comentarios
        # Penalización por baja popularidad
        if self.suscriptores < 500:
            return alcance / 2
        return alcance

In [4]:
class TikToker(Usuario):
    def __init__(self, user, comentarios, likes, views):
        super().__init__(user, comentarios, likes)
        self.views = views

    def alcance(self):
        # Cálculo basal
        alcance = self.likes * 4 - self.comentarios
        # Penalización por baja popularidad
        if self.views < 300:
            return -alcance * 3
        return alcance

Ahora debemos cargar la información de distintos personaes que se encuentra en la carpeta Data en el archivo ```influencers.csv```

In [5]:
import os

# Inicializamos listas vacías para cada tipo de influencer
instagramers = []
youtubers = []
tiktokers = []

# Ruta al archivo 
archivo_csv = os.path.join("data", "influencers.csv")


with open(archivo_csv, "r", encoding="utf-8") as archivo:
    archivo.readline()  # Saltar la cabecera
    for linea in archivo:
        usuario, comentarios, likes, extra, tipo = linea.strip().split(",")
        comentarios = int(comentarios)
        likes = int(likes)
        extra = int(extra)

        if tipo == "instagramer":
            instagramers.append(Instagramer(usuario, comentarios, likes, extra))
        elif tipo == "youtuber":
            youtubers.append(Youtuber(usuario, comentarios, likes, extra))
        elif tipo == "tiktoker":
            tiktokers.append(TikToker(usuario, comentarios, likes, extra))

Creamos una función para calcular la popularidad promedio de los influencers por cada red social

In [6]:
def popularidad(influencers):
    if not influencers:
        return 0
    total = 0
    for influencer in influencers:
        total += influencer.alcance()
    return total / len(influencers)

Finalmente Observamos los resultados y tomamos una desición:

In [7]:
# Calculamos la popularidad promedio por red social
popularidades = [
    (popularidad(instagramers), "Instagram"),
    (popularidad(youtubers), "YouTube"),
    (popularidad(tiktokers), "TikTok")
]

# Ordenamos de mayor a menor popularidad
popularidades.sort(reverse=True)

# Imprimimos decisión final
mejor_red = popularidades[0][1]
print(f"💡 La red social más conveniente para convertirte en influencer es: {mejor_red}")

💡 La red social más conveniente para convertirte en influencer es: Instagram


## @Decoradores

### Introducción (Muy breve)
En Python, **las funciones son objetos**. Esto significa que pueden ser pasadas como **argumentos, retornadas por otras funciones y modificadas por otras funciones**. Un **decorador** es simplemente una función que recibe como argumento otra función y retorna una función modificada


### Decoradores como composición de funciones
Una forma intuitiva de entender los **decoradores** en Python es compararlos con la **composición de funciones** en matemáticas.

En matemáticas, cuando tenemos dos funciones:

- `g(x) = x * 2`  
- `f(x) = x + 3`

Podemos componerlas como:  

```math
(f ∘ g)(x) = f(g(x)) = (x * 2) + 3

In [8]:
# versión funcional
def g(x):
    return x * 2

def f(func):
    def decorada(x):
        return func(x) + 3
    return decorada

f_o_g = f(g)  # f(g(x)) → primero g, luego f
print(f_o_g(5)) 

13


In [9]:
# Con Azucar sintáctico
def f(func):
    def decorada(x):
        return func(x) + 3
    return decorada

@f  # Al utilizar el decorador @f antes de la definición de g, estamos haciendo f(g(5))
def g(x):
    return x * 2

print(g(5))

13


### Decoradores con Parametros
En algunos casos es necesario darle paramatros a la función a decorar, para esto necesitamos agregarle otro nivel (`def`) para pasarle el parametro necesario

In [10]:
def derivada(h=1e-5):      # 1️⃣ Recibe el parámetro del decorador, h es el tamaño de la diferencia
    def decorador(func):         # 2️⃣ Recibe la función a decorar
        def derivada_aproximada(x):     # 3️⃣ Es la función decorada que será ejecutada
            return (func(x + h) - func(x)) / h
        return derivada_aproximada
    return decorador


@derivada()
def cuadrado(x):
    return x**2

@derivada()
def seno(x):
    from math import sin
    return sin(x)

print("f(x) = x² → f'(3) ≈", cuadrado(3))
print("f(x) = sin(x) → f'(π/2) ≈", seno(3.14159 / 2))

f(x) = x² → f'(3) ≈ 6.000009999951316
f(x) = sin(x) → f'(π/2) ≈ -3.6732061836630687e-06


In [11]:
def integral(a, b, n=1000):    # 1️⃣ Recibe el parámetro del decorador, n es el número de rectangulos
    def decorador(func):            # 2️⃣ Recibe la función a decorar
        def integral_aproximada():         # 3️⃣ Es la función decorada que será ejecutada
            h = (b - a) / n
            total = 0
            for i in range(n):
                x = a + i * h
                total += func(x) * h
            return total
        return integral_aproximada
    return decorador

In [12]:
from math import sin

@integral(0, 3.14159)
def senito(x):
    return sin(x)

print("∫ sin(x) dx de 0 a π ≈", senito())   # aproximadamente 1.99999 -> 2

∫ sin(x) dx de 0 a π ≈ 1.999998350896677


## Ejercicio: Pokémon Forever

En este ejercicio vamos a modelar un sistema de Pokémon iniciales usando programación orientada a objetos. Para garantizar que todas las clases compartan una estructura común, deberás crear una **clase abstracta** llamada `Pokemon`, que servirá como base para distintos tipos de Pokémon: **Agua**, **Fuego** y **Planta**.

###  Requerimientos:

#### 1. Clase abstracta `Pokemon`:

- Atributos:
  - `nombre`: nombre del Pokémon.
  - `nivel`: nivel del Pokémon (entero mayor o igual a 1).
  - `id`: identificador único incremental (compartido entre todos los Pokémon).
  - `_vida`: valor interno que representa la salud del Pokémon (entre 0 y 100).
  - `ataque_base`: valor base de daño.

- Propiedades:
  - `ataque`: calculado dinámicamente, con probabilidad de crítico si `random() < 0.5`.
  - `vida`: incluye `@setter` que limita la vida entre 0 y 100.

- Métodos:
  - `atacar(self, otro_pokemon)`: método **abstracto**, que cada subclase implementa con un mensaje distinto.  
    Dentro de este método se invoca `super().atacar()` para aplicar el daño real.
  - Métodos especiales `__str__`, `__repr__`, `__eq__`, y `__lt__` para mostrar, comparar y ordenar Pokémon.

---

#### 2. Clases concretas especializadas:

- `PokemonAgua(Pokemon)`
- `PokemonFuego(Pokemon)`
- `PokemonPlanta(Pokemon)`

Cada subclase:
- Redefine el método `atacar()` con un mensaje personalizado (e.g., `"usa HidroBomba!"`).
- Llama a `super().atacar(otro_pokemon)` para realizar el cálculo real de daño.

In [14]:
# Primero Importamos los módulos necesarios
from abc import ABC, abstractmethod
from random import random

In [15]:
# Clase abstracta
class Pokemon(ABC):
    identificador = 0  # Variable de clase

    def __init__(self, nombre, nivel):
        self.nombre = nombre
        self.nivel = nivel
        self.id = self.identificador
        Pokemon.identificador += 1

        self._vida = 100
        self.ataque_base = 10

    @property
    def ataque(self):
        probabilidad = random()
        if probabilidad < 0.5:
            return self.ataque_base * self.nivel
        return self.ataque_base
        
    @property
    def vida(self):
        return self._vida
    
    @vida.setter
    def vida(self, valor):
        self._vida = valor
        if self._vida < 0:
            self._vida = 0
        elif self._vida > 100:
            self._vida = 100


    @abstractmethod
    def atacar(self, otro_pokemon):
        damage = self.ataque
        otro_pokemon.vida -= damage
        print(f"El ataque fue de {damage} puntos de vida, ahora {otro_pokemon.nombre} tiene {otro_pokemon.vida} puntos de vida")

    def __str__(self):
        return f"{self.nombre} - Nivel {self.nivel}"
    
    def __repr__(self):
        return f"Pokemon N°{self.id}"

    def __eq__(self, other):
        return isinstance(other, Pokemon) and self.nombre == other.nombre and self.nivel == other.nivel

    def __lt__(self, other):
        return self.nivel < other.nivel

In [16]:
class PokemonAgua(Pokemon):

    def atacar(self, otro_pokemon):
        print(f"{self.nombre} utiliza HidroBomba contra {otro_pokemon.nombre}!")
        super().atacar(otro_pokemon)


class PokemonFuego(Pokemon):

    def atacar(self, otro_pokemon):
        print(f"{self.nombre} utiliza Lanzallamas! contra {otro_pokemon.nombre}!")
        super().atacar(otro_pokemon)

class PokemonPlanta(Pokemon):

    def atacar(self, otro_pokemon):
        print(f"{self.nombre} usa Rayo Solar contra {otro_pokemon.nombre}!")
        super().atacar(otro_pokemon)

In [17]:
# Pruebas
squirtle = PokemonAgua("Squirtle", 5)
charmander = PokemonFuego("Charmander", 6)
bulbasaur = PokemonPlanta("Bulbasaur", 5)

# __str__
print(squirtle)    # Squirtle (Agua) - Nivel 5
print(charmander)  # Charmander (Fuego) - Nivel 6
print(bulbasaur)   # Bulbasaur (Planta) - Nivel 5

Squirtle - Nivel 5
Charmander - Nivel 6
Bulbasaur - Nivel 5


In [18]:
# atacar
squirtle.atacar(charmander)   # Squirtle lanza Pistola Agua contra Charmander!
charmander.atacar(bulbasaur)  # Charmander lanza Ascuas contra Bulbasaur!
bulbasaur.atacar(squirtle)    # Bulbasaur usa Látigo Cepa contra Squirtle!

# Comparaciones
print(squirtle == bulbasaur)  # False
print(squirtle == PokemonAgua("Squirtle", 5))  # True
print(squirtle < charmander)  # True

Squirtle utiliza HidroBomba contra Charmander!
El ataque fue de 10 puntos de vida, ahora Charmander tiene 90 puntos de vida
Charmander utiliza Lanzallamas! contra Bulbasaur!
El ataque fue de 10 puntos de vida, ahora Bulbasaur tiene 90 puntos de vida
Bulbasaur usa Rayo Solar contra Squirtle!
El ataque fue de 10 puntos de vida, ahora Squirtle tiene 90 puntos de vida
False
True
True


Ahora debemos crear una estructura que permita almacenar nuestros pokemones atrapados, heradaremos de la estructura build-in ```dict```

In [19]:
class Pokedex(dict):
    def agregar(self, pokemon):
        # Agrega un Pokemon a la pokedex usando su nombre como clave
        self[pokemon.nombre] = pokemon

    def mostrar(self):
        # Mostramos todos los Pokemones
        print("📘 Tu Pokédex:")
        for nombre, pokemon in self.items():
            print(f"- {pokemon}")

    def __contains__(self, nombre):
        # Verifica si un Pokemon esta en la Pokedex
        return nombre in self.keys()

In [20]:
squirtle = PokemonAgua("Squirtle", 5)
charmander = PokemonFuego("Charmander", 6)
bulbasaur = PokemonPlanta("Bulbasaur", 5)

mi_pokedex = Pokedex()

mi_pokedex.agregar(squirtle)
mi_pokedex.agregar(charmander)
mi_pokedex.agregar(bulbasaur)

mi_pokedex.mostrar()

# ¿Tengo a Pikachu?
print(f"Tengo a Pikachu? {'Pikachu' in mi_pokedex}") 

# ¿Tengo a Squirtle?
print(f"Tengo a Squirtle? {'Squirtle' in mi_pokedex}")

# Acceder directamente por nombre
print(mi_pokedex["Bulbasaur"].nivel)  # 5

📘 Tu Pokédex:
- Squirtle - Nivel 5
- Charmander - Nivel 6
- Bulbasaur - Nivel 5
Tengo a Pikachu? False
Tengo a Squirtle? True
5


**¿Por que los siguientes prints son diferentes?**

In [21]:
for pokemon in mi_pokedex.values():
    print(pokemon)

Squirtle - Nivel 5
Charmander - Nivel 6
Bulbasaur - Nivel 5


In [22]:
print(mi_pokedex.values())

dict_values([Pokemon N°4, Pokemon N°5, Pokemon N°6])
